# Pillar 5: Structural Transformation (Lagged Effect)

Primary model:
agri_yield_it = beta1*mgnrega_lag1_it + rainfall + FE + epsilon

Secondary check:
income_it = beta1*mgnrega_lag1_it + rainfall + FE + epsilon

Primary metric: beta1 on lagged MGNREGA

In [1]:
from pathlib import Path
import pandas as pd
from analysis_utils import load_and_preprocess, fit_fe_clustered, minmax_normalize, compute_vif, summarize_model

DATA_PATH = Path('Panel_Data 2014-24.csv')
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

df = load_and_preprocess(str(DATA_PATH))

In [2]:
formula_yield = 'agri_yield ~ mgnrega_lag1 + rainfall + C(region_id) + C(year)'
formula_income = 'income ~ mgnrega_lag1 + rainfall + C(region_id) + C(year)'

res_yield, df_yield = fit_fe_clustered(formula_yield, df)
res_income, df_income = fit_fe_clustered(formula_income, df)

print('Lag to yield summary')
print(res_yield.summary())
print('Lag to income summary')
print(res_income.summary())

coef_table = pd.DataFrame([
    {'model': 'yield', 'term': 'mgnrega_lag1', 'coef': res_yield.params.get('mgnrega_lag1', 0.0), 'p_value': res_yield.pvalues.get('mgnrega_lag1')},
    {'model': 'income', 'term': 'mgnrega_lag1', 'coef': res_income.params.get('mgnrega_lag1', 0.0), 'p_value': res_income.pvalues.get('mgnrega_lag1')},
])
coef_table.to_csv(OUT_DIR / 'pillar5_coefficients.csv', index=False)
print(coef_table)

vif_yield = compute_vif(df_yield, ['mgnrega_lag1', 'rainfall'])
vif_income = compute_vif(df_income, ['mgnrega_lag1', 'rainfall'])
vif_yield.to_csv(OUT_DIR / 'pillar5_vif_yield.csv', index=False)
vif_income.to_csv(OUT_DIR / 'pillar5_vif_income.csv', index=False)
print(vif_yield)
print(vif_income)

beta_primary = res_yield.params.get('mgnrega_lag1', 0.0)
region_raw = df_yield.groupby('region_id')['mgnrega_lag1'].mean() * beta_primary
pillar5_score = minmax_normalize(region_raw).rename('pillar5_structural_score').reset_index()
pillar5_score.to_csv(OUT_DIR / 'pillar5_structural_score.csv', index=False)
pillar5_score.head()

Lag to yield summary
                            OLS Regression Results                            
Dep. Variable:             agri_yield   R-squared:                       0.882
Model:                            OLS   Adj. R-squared:                  0.868
Method:                 Least Squares   F-statistic:                     1630.
Date:                Mon, 13 Apr 2026   Prob (F-statistic):          3.01e-229
Time:                        22:34:18   Log-Likelihood:                -7979.8
No. Observations:                2560   AIC:                         1.649e+04
Df Residuals:                    2293   BIC:                         1.806e+04
Df Model:                         266                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------

C:\Users\Tanishq op\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 266, but rank is 11
  warnings.warn('covariance of constraints does not have full '
C:\Users\Tanishq op\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 266, but rank is 11
  warnings.warn('covariance of constraints does not have full '


,region_id,pillar5_structural_score
0,ADILABAD_TELANGANA,0.526388
1,AGRA_UTTAR PRADESH,0.745651
2,AJMER_RAJASTHAN,0.210702
3,ALAPPUZHA_KERALA,0.331930
4,ALIGARH_UTTAR PRADESH,0.787634
